In [3]:
# 🌟 Exercise 1: Duplicate Detection and Removal

# Step 1: Import required libraries
import pandas as pd

# Step 2: Load the Titanic dataset (make sure train.csv is in the same directory as this notebook)
df = pd.read_csv("train.csv")

# Step 3: Preview the dataset
print("🔍 Preview of the dataset:")
print(df.head())

# Step 4: Check the number of rows before removing duplicates
rows_before = df.shape[0]
print(f"\n📏 Number of rows before removing duplicates: {rows_before}")

# Step 5: Identify duplicate rows (rows that are exactly the same across all columns)
duplicate_rows = df.duplicated()
num_duplicates = duplicate_rows.sum()
print(f"🧯 Number of duplicate rows detected: {num_duplicates}")

# Step 6: Remove the duplicates from the dataset
df = df.drop_duplicates()

# Step 7: Confirm the new number of rows
rows_after = df.shape[0]
print(f"✅ Number of rows after removing duplicates: {rows_after}")
print(f"🧹 {rows_before - rows_after} duplicate rows removed.")

🔍 Preview of the dataset:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   

In [5]:
# 🌟 Exercise 2: Handling Missing Values

# Step 1: Identify missing values in each column
print("🔍 Missing values per column:")
print(df.isnull().sum())

# Step 2: Drop rows where 'Cabin' is missing (too many NaNs, likely not useful for this simple preprocessing)
df = df.drop(columns=['Cabin'])  # alternatively: df.dropna(subset=['Cabin'], inplace=True)

# Step 3: Fill missing values in 'Age' using the median (suitable for numerical/skewed data)
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

# Step 4: Fill missing values in 'Embarked' with the mode (most frequent value)
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

# Step 5: (Optional) If 'Fare' had missing values, we could fill with mean or median (but it doesn't in this dataset)
# df['Fare'] = df['Fare'].fillna(df['Fare'].mean())

# Step 6: Check that there are no more missing values
print("\n✅ Missing values after cleaning:")

🔍 Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

✅ Missing values after cleaning:


In [7]:
# 🌟 Exercise 3: Feature Engineering

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

# Step 1: Create a new feature 'FamilySize' from 'SibSp' (siblings/spouses) and 'Parch' (parents/children)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

# Step 2: Extract the title from the 'Name' column
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Optional: Group rare titles into a single 'Rare' category
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 
               'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Step 3: One-hot encode categorical features: 'Sex', 'Embarked', 'Title'
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# Step 4: Drop columns we no longer need after feature extraction
df = df.drop(columns=['Name', 'Ticket', 'PassengerId'])

# Step 5: Standardize numeric features if needed (example: 'Age', 'Fare', 'FamilySize')
scaler = StandardScaler()
numeric_cols = ['Age', 'Fare', 'FamilySize']
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Step 6: Preview the transformed dataset
print("✅ Transformed dataset (first 5 rows):")
print(df.head())

<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\.'
/var/folders/98/72mf16fn6sv4h6j6pj3w1__h0000gn/T/ipykernel_97139/3402679875.py:9: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


✅ Transformed dataset (first 5 rows):
   Survived  Pclass       Age  SibSp  Parch      Fare  FamilySize  Sex_male  \
0         0       3 -0.565736      1      0 -0.502445    0.059160      True   
1         1       1  0.663861      1      0  0.786845    0.059160     False   
2         1       3 -0.258337      0      0 -0.488854   -0.560975     False   
3         1       1  0.433312      1      0  0.420730    0.059160     False   
4         0       3  0.433312      0      0 -0.486337   -0.560975      True   

   Embarked_Q  Embarked_S  Title_Miss  Title_Mr  Title_Mrs  Title_Rare  
0       False        True       False      True      False       False  
1       False       False       False     False       True       False  
2       False        True        True     False      False       False  
3       False        True       False     False       True       False  
4       False        True       False      True      False       False  


In [9]:
# 🌟 Exercise 4: Outlier Detection and Handling

import numpy as np
from scipy import stats

# Step 1: Detect outliers using the IQR method for 'Fare' and 'Age'
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return series[(series < lower_bound) | (series > upper_bound)]

# Step 2: Apply the function to 'Fare' and 'Age'
fare_outliers = detect_outliers_iqr(df['Fare'])
age_outliers = detect_outliers_iqr(df['Age'])

print(f"🚨 Number of Fare outliers (IQR): {fare_outliers.shape[0]}")
print(f"🚨 Number of Age outliers (IQR): {age_outliers.shape[0]}")

# Step 3: Handle outliers – we will cap the outliers using the IQR bounds (Winsorization)
def cap_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.where(series < lower_bound, lower_bound,
           np.where(series > upper_bound, upper_bound, series))

df['Fare'] = cap_outliers_iqr(df['Fare'])
df['Age'] = cap_outliers_iqr(df['Age'])

# Optional: Check for extreme outliers using Z-score method (standardized data)
z_scores_fare = np.abs(stats.zscore(df['Fare']))
z_scores_age = np.abs(stats.zscore(df['Age']))

print(f"📊 Number of extreme Fare outliers (Z-score > 3): {(z_scores_fare > 3).sum()}")
print(f"📊 Number of extreme Age outliers (Z-score > 3): {(z_scores_age > 3).sum()}")

# Step 4: Final preview
print("\n✅ Outliers handled. Data preview:")
print(df[['Fare', 'Age']].describe())

🚨 Number of Fare outliers (IQR): 116
🚨 Number of Age outliers (IQR): 66
📊 Number of extreme Fare outliers (Z-score > 3): 0
📊 Number of extreme Age outliers (Z-score > 3): 0

✅ Outliers handled. Data preview:
             Fare         Age
count  891.000000  891.000000
mean    -0.164247   -0.024769
std      0.412391    0.927737
min     -0.648422   -2.064308
25%     -0.489148   -0.565736
50%     -0.357391   -0.104637
75%     -0.024246    0.433312
max      0.673106    1.931883


In [11]:
# 🌟 Exercise 5: Data Standardization and Normalization

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Step 1: Check distribution and range of numeric features
print("🔍 Numeric features before scaling:")
print(df[['Age', 'Fare', 'FamilySize']].describe())

# Step 2: Apply Standardization (mean = 0, std = 1) to 'Age' and 'Fare'
standard_scaler = StandardScaler()
df[['Age_std', 'Fare_std']] = standard_scaler.fit_transform(df[['Age', 'Fare']])

# Step 3: Apply Normalization (scaled to [0, 1]) to 'FamilySize'
minmax_scaler = MinMaxScaler()
df[['FamilySize_norm']] = minmax_scaler.fit_transform(df[['FamilySize']])

# Step 4: Preview the standardized and normalized values
print("\n✅ After Standardization and Normalization:")
print(df[['Age_std', 'Fare_std', 'FamilySize_norm']].head())

🔍 Numeric features before scaling:
              Age        Fare    FamilySize
count  891.000000  891.000000  8.910000e+02
mean    -0.024769   -0.164247 -2.392400e-17
std      0.927737    0.412391  1.000562e+00
min     -2.064308   -0.648422 -5.609748e-01
25%     -0.565736   -0.489148 -5.609748e-01
50%     -0.104637   -0.357391 -5.609748e-01
75%      0.433312   -0.024246  5.915988e-02
max      1.931883    0.673106  5.640372e+00

✅ After Standardization and Normalization:
    Age_std  Fare_std  FamilySize_norm
0 -0.583432 -0.820552              0.1
1  0.742685  2.031623              0.1
2 -0.251903 -0.787578              0.0
3  0.494038  1.419297              0.1
4  0.494038 -0.781471              0.0


In [13]:
# 🌟 Exercise 6: Feature Encoding

from sklearn.preprocessing import LabelEncoder

# Step 1: Check remaining categorical columns (if any were not encoded earlier)
print("🔍 Columns in dataset:")
print(df.columns)

# Example setup (if you've reset the notebook): let's assume 'Sex' and 'Embarked' are still not encoded
# You can skip this block if you already did it in Exercise 3

# Step 2: Label encode 'Pclass' (ordinal categorical variable)
# Note: Pclass is actually already numeric, but it's an ordinal feature (1st, 2nd, 3rd class)
# No transformation needed unless using tree-based models — just ensure it is treated as categorical when modeling

# Step 3: Label encode 'Embarked' as an ordinal variable (only if not one-hot encoded)
# If 'Embarked' is still in original string format:
if 'Embarked' in df.columns and df['Embarked'].dtype == 'object':
    le = LabelEncoder()
    df['Embarked_Encoded'] = le.fit_transform(df['Embarked'])

# Step 4: One-hot encode nominal variable 'Sex' (if not already done)
# If column 'Sex' exists and is not already encoded
if 'Sex' in df.columns and df['Sex'].dtype == 'object':
    df = pd.get_dummies(df, columns=['Sex'], drop_first=True)

# Step 5: Final check
print("\n✅ Encoded dataset preview:")
print(df.head())

🔍 Columns in dataset:
Index(['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize',
       'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr',
       'Title_Mrs', 'Title_Rare', 'Age_std', 'Fare_std', 'FamilySize_norm'],
      dtype='object')

✅ Encoded dataset preview:
   Survived  Pclass       Age  SibSp  Parch      Fare  FamilySize  Sex_male  \
0         0       3 -0.565736      1      0 -0.502445    0.059160      True   
1         1       1  0.663861      1      0  0.673106    0.059160     False   
2         1       3 -0.258337      0      0 -0.488854   -0.560975     False   
3         1       1  0.433312      1      0  0.420730    0.059160     False   
4         0       3  0.433312      0      0 -0.486337   -0.560975      True   

   Embarked_Q  Embarked_S  Title_Miss  Title_Mr  Title_Mrs  Title_Rare  \
0       False        True       False      True      False       False   
1       False       False       False     False       True       False   
2    

In [15]:
# 🌟 Exercise 7: Data Transformation for Age Feature

# Step 1: Check the distribution of the 'Age' column
print("🔍 Age column statistics:")
print(df['Age'].describe())

# Step 2: Create age bins (categories) using pd.cut()
# We'll define age groups: Child, Teen, Young Adult, Adult, Senior
age_bins = [-1, 12, 18, 30, 60, 100]
age_labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']

df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels)

# Step 3: One-hot encode the new AgeGroup column
df = pd.get_dummies(df, columns=['AgeGroup'])

# Step 4: Final preview of the age groups
print("\n✅ Dataset with AgeGroup one-hot encoded:")
print(df.filter(like='AgeGroup').head())

🔍 Age column statistics:
count    891.000000
mean      -0.024769
std        0.927737
min       -2.064308
25%       -0.565736
50%       -0.104637
75%        0.433312
max        1.931883
Name: Age, dtype: float64

✅ Dataset with AgeGroup one-hot encoded:
   AgeGroup_Child  AgeGroup_Teen  AgeGroup_YoungAdult  AgeGroup_Adult  \
0            True          False                False           False   
1            True          False                False           False   
2            True          False                False           False   
3            True          False                False           False   
4            True          False                False           False   

   AgeGroup_Senior  
0            False  
1            False  
2            False  
3            False  
4            False  
